In [ ]:
!pip3 install torch torchaudio torchvision torchtext torchdata
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 2.1 MB/s eta 0:00:00


In [1]:
vocabulary = {
    "gato": 0,
    "perro": 1,
    "elefante": 2,
    "hipopotamo": 3,
    "palmera": 4,
    "helecho": 5,
    "rosa": 6,
    "gardenia": 7
}

classes = {
    "animal": 0,
    "plant": 1
}

translate = ["animal", "plant"]

In [2]:
from torch import nn
import torch

class WordClassifier(nn.Module):
  def __init__(self, vocabulary, classes) -> None:
    super().__init__()
    self.embeddings_layer = nn.Embedding(len(vocabulary), 8)
    self.linear_relu_stack = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, len(classes)),
        )

  def forward(self, x):
    return self.linear_relu_stack(self.embeddings_layer(x))


In [10]:
wc = WordClassifier(vocabulary, classes)
output = wc(torch.tensor([1, 2], dtype=torch.long).unsqueeze(0))
_, prediction = torch.max(output.data, 1)
for p in prediction.numpy()[0]:
  print(translate[p])

plant
plant


In [51]:
tokens = [
    "gato","perro","elefante","hipopotamo","palmera",
    "helecho","rosa","gardenia","gato","perro","elefante",
    "hipopotamo","palmera","helecho","rosa","gardenia","gato",
    "palmera","elefante","rosa"
]

classification = [
    "animal","animal","animal","animal",
    "plant","plant","plant","plant","animal","animal",
    "animal","animal","plant","plant","plant","plant",
    "animal","plant","animal","plant"
]

In [16]:
indexs = [vocabulary[t] for t in tokens]
target = [classes[i] for i in classification]

indexs = torch.tensor(indexs, dtype=torch.long)
target = torch.tensor(target, dtype=torch.long)

In [13]:
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split

train_data, test_data = random_split(TensorDataset(indexs, target), [0.7, 0.3])
train_loader = DataLoader(dataset=train_data, batch_size=2, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=2, shuffle=True)

In [14]:
from tqdm import tqdm
from torch import optim

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(wc.parameters())

epochs = 100
for epoch in tqdm(range(epochs)):
  wc.train()
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data
    optimizer.zero_grad()
    outputs = wc(inputs)

    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer.step()

correct = 0
total = 0

wc.eval()
with torch.no_grad():
  for data in test_loader:
    inputs, labels = data
    outputs = wc(inputs)
    _, prediction = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (prediction == labels).sum().item()

print(f" Accuracy: {correct / total}")


100%|██████████| 100/100 [00:00<00:00, 114.90it/s]

Accuracy: 1.0


In [19]:
word = [vocabulary["helecho"]]
_, prediction = torch.max(wc(torch.tensor(word, dtype=torch.long)), 1)
translate[prediction.tolist()[0]]

'plant'

Clasificación por secuencias de texto

In [20]:
def tokenizer(phrase, vocabulary):
  tokens_phrase = []
  for word in phrase.lower().split():
    if word in vocabulary:
      tokens_phrase.append(vocabulary[word])
    else:
      tokens_phrase.append(len(vocabulary) - 1)
  return torch.tensor(tokens_phrase, dtype=torch.long).unsqueeze(0)


In [37]:
emails = [
    # 0-9
    "Hola equipo adjunto el reporte de ventas solicitado para la reunión de mañana",
    "Su cuenta será bloqueada ingrese su contraseña para evitar la suspensión",
    "Buenos días confirmo la recepción del documento y agradezco la información",
    "Felicidades ha ganado un premio reclame enviando sus datos bancarios",
    "Les comparto la presentación del proyecto para recibir sus comentarios",
    "Actualice su cuenta mediante este enlace para recuperar el acceso",
    "La reunión se realizará el lunes a las diez en la sala de conferencias",
    "Detectamos actividad sospechosa verifique su identidad proporcionando su contraseña",
    "Adjunto la factura correspondiente al servicio contratado quedo atento a sus dudas",
    "Su paquete está retenido pague la tarifa pendiente para evitar su devolución",

    # 10-19
    "Confirmo mi asistencia a la reunión programada para el próximo jueves",
    "Su tarjeta fue bloqueada introduzca su número y código de seguridad",
    "El informe final está disponible en la carpeta compartida del equipo",
    "Reclame su premio exclusivo pagando una pequeña tarifa de envío",
    "Adjunto los resultados de las pruebas realizadas durante esta semana",
    "Su contraseña expiró responda este correo con sus credenciales para renovarla",
    "El profesor publicó las calificaciones en la plataforma institucional",
    "Su cuenta bancaria será cerrada confirme sus datos personales inmediatamente",
    "Gracias por compartir las minutas de la reunión de ayer",
    "Un depósito está pendiente envíe su número de cuenta para liberarlo",

    # 20-29
    "La factura electrónica fue enviada al correo registrado en el sistema",
    "Ha sido seleccionado para recibir dinero gratis complete este formulario",
    "El mantenimiento del servidor comenzará el sábado por la noche",
    "Su servicio será cancelado pague ahora mediante este enlace externo",
    "Confirmo la recepción de los archivos adjuntos sin inconvenientes",
    "Detectamos un acceso irregular comparta el código recibido en su teléfono",
    "La convocatoria del congreso está disponible en la página de la universidad",
    "Obtenga un préstamo inmediato pagando una comisión por adelantado",
    "El pedido fue entregado correctamente en la dirección registrada",
    "Su buzón está lleno ingrese su contraseña para ampliar el almacenamiento",

    # 30-39
    "La sesión de capacitación iniciará mañana a las nueve de la mañana",
    "Su cuenta presenta una deuda urgente descargue el archivo para pagar",
    "Adjunto el presupuesto actualizado para su revisión y aprobación",
    "Su paquete no pudo entregarse proporcione los datos de su tarjeta",
    "La solicitud de vacaciones fue recibida por el departamento administrativo",
    "Hemos detectado un virus instale este programa para proteger su equipo",
    "El laboratorio permanecerá cerrado durante el periodo de mantenimiento",
    "Su reembolso está autorizado ingrese sus datos bancarios para recibirlo",
    "La reunión de seguimiento fue reprogramada para el próximo miércoles",
    "Su identidad está en riesgo envíe una copia de su identificación",

    # 40-49
    "El sistema generó correctamente el respaldo automático de los archivos",
    "Su cuenta será eliminada confirme su contraseña en el siguiente enlace",
    "Gracias por enviar los documentos solicitados para el proceso de admisión",
    "Ha ganado un teléfono nuevo pague el envío para recibir su regalo",
    "El reporte mensual incluye los indicadores de calidad y productividad",
    "Su nómina está retenida proporcione sus datos bancarios para liberarla",
    "La sesión de asesoría se realizará mediante la plataforma institucional",
    "El banco necesita verificar su NIP responda este correo inmediatamente",
    "La orden de compra fue autorizada por el departamento correspondiente",
    "Reciba una herencia millonaria enviando sus datos personales y bancarios",

    # 50-59
    "El calendario académico actualizado está disponible en el portal universitario",
    "Su suscripción será cancelada confirme sus datos de pago para renovarla",
    "Adjunto la lista de materiales necesarios para la práctica de laboratorio",
    "Su tarjeta tiene un cargo pendiente ingrese aquí para evitar una multa",
    "El comité revisará la propuesta durante la próxima semana",
    "Descargue el documento adjunto y habilite las macros para evitar una sanción",
    "Su inscripción al seminario ha sido confirmada satisfactoriamente",
    "El director solicita una transferencia urgente mantenga esta operación en secreto",
    "El contrato firmado fue recibido por el área jurídica",
    "Su licencia será desactivada instale esta actualización no verificada",

    # 60-69
    "El departamento financiero publicó el calendario de pagos a proveedores",
    "Su cuenta de correo será suspendida valide su identidad en este enlace",
    "La factura del servicio está disponible en el portal oficial de clientes",
    "Reciba una recompensa exclusiva proporcionando su contraseña y dirección personal",
    "El informe técnico contiene las gráficas obtenidas durante las pruebas",
    "Su transferencia fue bloqueada envíe el código de seguridad para liberarla",
    "Confirmamos la recepción de su solicitud de constancia de estudios",
    "Pague los impuestos de su paquete mediante este enlace no oficial",
    "La actualización del sistema terminó correctamente durante el mantenimiento",
    "Su cuenta necesita una actualización urgente confirme su NIP bancario",

    # 70-79
    "El horario de atención del laboratorio será de ocho a cuatro",
    "Su pago fue rechazado introduzca los datos completos de su tarjeta",
    "Le comparto el temario y los materiales para la próxima sesión",
    "Evite el cierre definitivo de su cuenta ingresando su contraseña ahora",
    "El comité acordó revisar nuevamente los resultados del proyecto",
    "Ha sido elegido para un bono especial envíe su cuenta bancaria",
    "La documentación para el trámite fue recibida correctamente",
    "Su computadora está infectada pague el rescate para recuperar sus archivos",
    "La reunión de coordinación comenzará quince minutos más tarde",
    "Su cuenta de compras tiene puntos vencidos registre su tarjeta aquí",

    # 80-89
    "El recibo de electricidad está disponible en el portal de clientes",
    "Su préstamo está aprobado pague una comisión anticipada para recibirlo",
    "Confirmo que recibí el paquete y que todo llegó en buen estado",
    "Su cuenta institucional será cerrada responda con su usuario y contraseña",
    "La encuesta estará disponible hasta el viernes por la tarde",
    "Su devolución de impuestos está lista proporcione sus datos para cobrarla",
    "El departamento de soporte recibió su solicitud y dará seguimiento",
    "Comparta el código de autenticación para evitar el bloqueo de su cuenta",
    "El calendario de exámenes fue publicado en la plataforma académica",
    "Su servicio será desconectado hoy pague inmediatamente en este enlace",

    # 90-99
    "Adjunto la versión corregida del documento para su revisión final",
    "El banco detectó un problema confirme su contraseña y código de seguridad",
    "La nómina correspondiente a esta quincena ya fue procesada",
    "Reclame el premio que ganó enviando una copia de su identificación",
    "El mantenimiento preventivo del equipo se realizará el próximo martes",
    "Su cuenta presenta actividad sospechosa instale esta aplicación de acceso remoto",
    "Gracias por participar en la reunión de seguimiento del proyecto",
    "Su paquete internacional está detenido pague ahora una tarifa de liberación",
    "La solicitud de soporte quedó registrada con su número de seguimiento",
    "Su cuenta será eliminada en diez minutos confirme sus credenciales inmediatamente"
]

labels = [
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
    0, 1, 0, 1, 0, 1, 0, 1, 0, 1
]

# 0 = legítimo
# 1 = malicioso

classes = ["Legítimo", "Malicioso"]

In [38]:
vocabulary = {}
for email in emails:
    for word in email.lower().split():
        if word not in vocabulary:
            vocabulary[word] = len(vocabulary) + 1

vocabulary["<PAD>"] = 0
vocabulary["<UNK>"] = len(vocabulary)

print(vocabulary)
print("Tamaño del vocabulario:", len(vocabulary))

{'hola': 1, 'equipo': 2, 'adjunto': 3, 'el': 4, 'reporte': 5, 'de': 6, 'ventas': 7, 'solicitado': 8, 'para': 9, 'la': 10, 'reunión': 11, 'mañana': 12, 'su': 13, 'cuenta': 14, 'será': 15, 'bloqueada': 16, 'ingrese': 17, 'contraseña': 18, 'evitar': 19, 'suspensión': 20, 'buenos': 21, 'días': 22, 'confirmo': 23, 'recepción': 24, 'del': 25, 'documento': 26, 'y': 27, 'agradezco': 28, 'información': 29, 'felicidades': 30, 'ha': 31, 'ganado': 32, 'un': 33, 'premio': 34, 'reclame': 35, 'enviando': 36, 'sus': 37, 'datos': 38, 'bancarios': 39, 'les': 40, 'comparto': 41, 'presentación': 42, 'proyecto': 43, 'recibir': 44, 'comentarios': 45, 'actualice': 46, 'mediante': 47, 'este': 48, 'enlace': 49, 'recuperar': 50, 'acceso': 51, 'se': 52, 'realizará': 53, 'lunes': 54, 'a': 55, 'las': 56, 'diez': 57, 'en': 58, 'sala': 59, 'conferencias': 60, 'detectamos': 61, 'actividad': 62, 'sospechosa': 63, 'verifique': 64, 'identidad': 65, 'proporcionando': 66, 'factura': 67, 'correspondiente': 68, 'al': 69, 's

In [41]:
import torch
from torch.nn.utils.rnn import pad_sequence

tokens = []
for email in emails:
  tokens_phrase = []
  for palabra in email.lower().split():
    tokens_phrase.append(vocabulary[palabra])
  tokens.append(torch.tensor(tokens_phrase, dtype=torch.long))

tokens = pad_sequence(
    tokens,
    batch_first=True,
    padding_value=0
)

print("Tokens con padding:")
print(tokens)

Tokens con padding:
tensor([[  1,   2,   3,  ...,   6,  12,   0],
        [ 13,  14,  15,  ...,   0,   0,   0],
        [ 21,  22,  23,  ...,   0,   0,   0],
        ...,
        [ 13,  75, 382,  ...,   0,   0,   0],
        [ 10, 196,   6,  ...,   0,   0,   0],
        [ 13,  14,  15,  ...,   0,   0,   0]])


In [42]:
class WordClassifierSeq(nn.Module):
  def __init__(self, vocabulary, classes) -> None:
    super().__init__()
    self.embeddings_layer = nn.Embedding(len(vocabulary), 8, padding_idx=0)
    self.linear_relu_stack = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, len(classes)),
        )

  def forward(self, x):
        mask = (x != 0).unsqueeze(-1)
        embeddings = self.embeddings_layer(x)
        embeddings = embeddings * mask
        suma = embeddings.sum(dim=1)
        cantidad = mask.sum(dim=1).clamp(min=1)
        pooled = suma / cantidad

        return self.linear_relu_stack(pooled)

In [43]:
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split

labels = torch.tensor(labels, dtype=torch.long)
train_data, test_data = random_split(TensorDataset(tokens, labels), [0.8, 0.2])
train_loader = DataLoader(dataset=train_data, batch_size=2, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=2, shuffle=True)

In [45]:
from tqdm import tqdm
from torch import optim

model = WordClassifierSeq(vocabulary, classes)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

epochs = 1000
for epoch in tqdm(range(epochs)):
  model.train()
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data
    optimizer.zero_grad()
    outputs = model(inputs)

    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer.step()

correct = 0
total = 0

model.eval()
with torch.no_grad():
  for data in test_loader:
    inputs, labels = data
    outputs = model(inputs)
    _, prediction = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (prediction == labels).sum().item()

print(f"Exactitud: {correct / total}")

100%|██████████| 1000/1000 [01:01<00:00, 16.18it/s]

Exactitud: 0.95


In [46]:
f = "El banco detectó un problema envie su numero de cuenta para resolverlo"
t = tokenizer(f, vocabulary)
_, pred = torch.max(model(t).data, 1)
print(classes[pred])

Malicioso


In [47]:
from torch.nn.utils.rnn import pack_padded_sequence
class WordClassifierRecurrent(nn.Module):
  def __init__(self, vocabulary, classes) -> None:
    super().__init__()
    self.embeddings_layer = nn.Embedding(len(vocabulary), 8, padding_idx=0)
    self.gru = nn.GRU(8, 16, batch_first=True)
    self.linear_relu_stack = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, len(classes)),
        )

  def forward(self, x):
        lengths = (x != 0).sum(dim=1).clamp(min=1)
        embeddings = self.embeddings_layer(x)
        packed = pack_padded_sequence(
            embeddings,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, hidden = self.gru(packed)
        representation = hidden[-1]

        return self.linear_relu_stack(representation)

In [48]:
from tqdm import tqdm
from torch import optim

model = WordClassifierRecurrent(vocabulary, classes)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

epochs = 1000
for epoch in tqdm(range(epochs)):
  model.train()
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data
    optimizer.zero_grad()
    outputs = model(inputs)

    loss = loss_fn(outputs, labels)
    loss.backward()
    optimizer.step()

correct = 0
total = 0

model.eval()
with torch.no_grad():
  for data in test_loader:
    inputs, labels = data
    outputs = model(inputs)
    _, prediction = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (prediction == labels).sum().item()

print(f"Accuracy: {correct / total}")

100%|██████████| 1000/1000 [03:41<00:00,  4.52it/s]


Accuracy: 1.0


In [49]:
f = "El recibo está disponible"
t = tokenizer(f, vocabulary)
_, pred = torch.max(model(t).data, 1)
print(classes[pred])

Legítimo


In [54]:
f = "Instale la siguiente aplicación y reciba un premio"
t = tokenizer(f, vocabulary)
_, pred = torch.max(model(t).data, 1)
print(classes[pred])

Malicioso
